In [24]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

# Параметры
CSV_PATH = "./data/emails_clean.csv"
TEXT_COL = "clean_text"   # столбец с текстом (твой)
TARGET_COL = "spam"       # столбец с меткой (твой)
K = 15

# 1) Загрузка
df = pd.read_csv(CSV_PATH)

# 3) Кодирование целевой переменной (если она строковая)
le_target = LabelEncoder()
y = le_target.fit_transform(df[TARGET_COL])

# 4) Векторизация текста (TF-IDF)
vec = TfidfVectorizer(max_features=20000, ngram_range=(1,2))  # параметры можно менять
X = vec.fit_transform(df[TEXT_COL])  # scipy sparse matrix

# 5) Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# 6) Модель KNN (sklearn) — поддерживает sparse вход
model = KNeighborsClassifier(n_neighbors=K, n_jobs=-1)
model.fit(X_train, y_train)

# 7) Оценка
y_pred = model.predict(X_test)
print("Accuracy on test:", accuracy_score(y_test, y_pred))

# 8) Функция для интерактивной классификации одного текста
def classify_text(text: str) -> str:
    vec_x = vec.transform([text])            # превращаем строку в вектор
    pred = model.predict(vec_x)             # получаем метку в numeric
    return le_target.inverse_transform(pred)[0]  # возвращаем исходное имя класса

# 9) Простой REPL для ввода текста пользователем
if __name__ == "__main__":
    print("\nВвод нового сообщения для классификации (пустая строка — выход):")
    while True:
        s = input("\nВведите текст письма: ").strip()
        if s == "":
            print("Выход.")
            break
        print("Алгоритм отнёс к классу:", classify_text(s))

Accuracy on test: 0.9837114601512508

Ввод нового сообщения для классификации (пустая строка — выход):
Алгоритм отнёс к классу: 1
Алгоритм отнёс к классу: 1
Алгоритм отнёс к классу: 1
Алгоритм отнёс к классу: 1
Алгоритм отнёс к классу: 1
Алгоритм отнёс к классу: 0
Алгоритм отнёс к классу: 0
Алгоритм отнёс к классу: 0
Алгоритм отнёс к классу: 0
Алгоритм отнёс к классу: 0
Выход.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

# ------------------------------
# Загружаем данные
# ------------------------------
data = pd.read_csv("./data/emails_clean.csv")

if not {"clean_text", "spam"}.issubset(data.columns):
    raise ValueError(f"Ожидались столбцы 'clean_text' и 'spam'. Найдено: {data.columns.tolist()}")

X_raw = data["clean_text"].fillna("").astype(str)   # тексты
y = data["spam"].values                             # целевая переменная (0 = не спам, 1 = спам)

# ------------------------------
# Векторизация текста
# ------------------------------
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X = vectorizer.fit_transform(X_raw)

# ------------------------------
# Разделение на train/test
# ------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# ------------------------------
# Обучение модели KNN
# ------------------------------
knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
knn.fit(X_train, y_train)

# ------------------------------
# Проверка на тестовых данных
# ------------------------------
y_pred = knn.predict(X_test)
print("Точность модели:", accuracy_score(y_test, y_pred))

# ------------------------------
# Мини-приложение для проверки
# ------------------------------
print("\n=== Проверка нового сообщения ===")
print("Введите текст письма (пустая строка — выход):")

while True:
    msg = input("\nСообщение: ").strip()
    if msg == "":
        print("Выход.")
        break
    vec_msg = vectorizer.transform([msg])
    pred = knn.predict(vec_msg)[0]
    print("Результат:", "СПАМ 🚨" if pred == 1 else "НЕ СПАМ ✅")

Точность модели: 0.9720767888307156

=== Проверка нового сообщения ===
Введите текст письма (пустая строка — выход):
Результат: СПАМ 🚨
Результат: НЕ СПАМ ✅
Результат: СПАМ 🚨
Результат: СПАМ 🚨
Результат: СПАМ 🚨
Результат: НЕ СПАМ ✅
Результат: НЕ СПАМ ✅
Результат: НЕ СПАМ ✅
Результат: НЕ СПАМ ✅
Результат: НЕ СПАМ ✅
Выход.
